# Flipkart GridLock Hackathon: Traffic Demand Prediction
**Author**: Pair Programming with Antigravity AI

This notebook contains a complete, production-grade machine learning pipeline to solve the traffic demand forecasting challenge.

### Solution Architecture Overview:
1. **Spatial Features**: Decode `geohash` codes to physical Latitude/Longitude coordinates.
2. **Temporal Features**: Convert time string timestamps into continuous minutes-from-midnight and cyclical sine/cosine representations.
3. **Historical Demand Matching**: Extract the exact matching spatial-temporal traffic demand from the previous full day (Day 48).
4. **Active Trend Deviation**: Compute the localized traffic volume growth/decline ratio between Day 49 and Day 48 morning hours to capture holiday or weather effects.
5. **5-Fold Cross-Validation**: Train LightGBM regressors across folds to produce robust, variance-reduced ensembled predictions.

## 1. Setup & Dependencies

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as gh
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import os
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
print('Dependencies loaded successfully!')

## 2. Advanced Feature Engineering & Preprocessing
Here we define our advanced preprocessing logic, which extracts structural spatial, temporal, and historical features:

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as gh

def load_and_preprocess_data(train_path="dataset/train.csv", test_path="dataset/test.csv"):
    print("Loading data...")
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    
    # Identify test indices for splitting back later
    train['is_test'] = 0
    test['is_test'] = 1
    test['demand'] = np.nan
    
    # Combine datasets for unified preprocessing
    full_df = pd.concat([train, test], ignore_index=True)
    
    # 1. Temporal features
    print("Parsing timestamps...")
    def ts_to_minutes(ts):
        h, m = map(int, ts.split(':'))
        return h * 60 + m
    
    full_df['minutes'] = full_df['timestamp'].apply(ts_to_minutes)
    full_df['hour'] = full_df['minutes'] / 60.0
    
    # Cyclical trigonometric features for time of day
    full_df['sin_time'] = np.sin(2 * np.pi * full_df['minutes'] / 1440.0)
    full_df['cos_time'] = np.cos(2 * np.pi * full_df['minutes'] / 1440.0)
    
    # Additional temporal features
    full_df['is_peak_hour'] = ((full_df['hour'] >= 7) & (full_df['hour'] <= 10)) | ((full_df['hour'] >= 17) & (full_df['hour'] <= 19))
    full_df['is_night'] = (full_df['hour'] < 6) | (full_df['hour'] >= 22)
    full_df['is_morning'] = (full_df['hour'] >= 6) & (full_df['hour'] < 12)
    full_df['is_afternoon'] = (full_df['hour'] >= 12) & (full_df['hour'] < 17)
    full_df['is_evening'] = (full_df['hour'] >= 17) & (full_df['hour'] < 22)
    
    # 2. Geohash decoding (Spatial features)
    print("Decoding geohashes...")
    # Cache geohash coordinates to speed up computation
    unique_geohashes = full_df['geohash'].unique()
    geohash_coords = {}
    for g in unique_geohashes:
        try:
            lat, lon = gh.decode(g)
            geohash_coords[g] = (lat, lon)
        except Exception:
            geohash_coords[g] = (np.nan, np.nan)
            
    full_df['latitude'] = full_df['geohash'].map(lambda g: geohash_coords[g][0])
    full_df['longitude'] = full_df['geohash'].map(lambda g: geohash_coords[g][1])
    
    # 3. Previous Day Demand Lookup (demand_last_day)
    print("Creating historical lag features...")
    # Group train day 48 demand by geohash and timestamp
    df_48 = train[train['day'] == 48]
    day48_lookup = df_48.groupby(['geohash', 'timestamp'])['demand'].mean().reset_index()
    day48_lookup.rename(columns={'demand': 'demand_last_day'}, inplace=True)
    
    # Also compute a geohash-level historical mean for fallback imputation
    geohash_day48_mean = df_48.groupby('geohash')['demand'].mean().reset_index()
    geohash_day48_mean.rename(columns={'demand': 'geohash_day48_mean'}, inplace=True)
    
    # Compute timestamp-level statistics (temporal pattern across all geohashes)
    timestamp_day48_mean = df_48.groupby('timestamp')['demand'].mean().reset_index()
    timestamp_day48_mean.rename(columns={'demand': 'timestamp_day48_mean'}, inplace=True)
    timestamp_day48_std = df_48.groupby('timestamp')['demand'].std().reset_index()
    timestamp_day48_std.rename(columns={'demand': 'timestamp_day48_std'}, inplace=True)
    
    # Compute road-type level patterns
    roadtype_day48_mean = df_48.groupby('RoadType')['demand'].mean().reset_index()
    roadtype_day48_mean.rename(columns={'demand': 'roadtype_day48_mean'}, inplace=True)
    
    # Merge back to full_df
    full_df = pd.merge(full_df, day48_lookup, on=['geohash', 'timestamp'], how='left')
    full_df = pd.merge(full_df, geohash_day48_mean, on='geohash', how='left')
    full_df = pd.merge(full_df, timestamp_day48_mean, on='timestamp', how='left')
    full_df = pd.merge(full_df, timestamp_day48_std, on='timestamp', how='left')
    full_df = pd.merge(full_df, roadtype_day48_mean, on='RoadType', how='left')
    
    # Fill missing day-48 lookups with the geohash's average demand from Day 48, then with global mean
    global_day48_mean = df_48['demand'].mean()
    full_df['demand_last_day'] = full_df['demand_last_day'].fillna(full_df['geohash_day48_mean'])
    full_df['demand_last_day'] = full_df['demand_last_day'].fillna(full_df['timestamp_day48_mean'])
    full_df['demand_last_day'] = full_df['demand_last_day'].fillna(global_day48_mean)
    
    # Fill timestamp stats with global if missing
    global_timestamp_std = df_48['demand'].std()
    full_df['timestamp_day48_std'] = full_df['timestamp_day48_std'].fillna(global_timestamp_std)
    full_df['roadtype_day48_mean'] = full_df['roadtype_day48_mean'].fillna(global_day48_mean)
    
    # 4. Day 49 Morning Trend Ratio
    # Compute trend of demand on day 49 compared to day 48 between 00:00 and 02:00
    print("Computing Day 49 morning trend ratio...")
    morning_timestamps = ['0:0', '0:15', '0:30', '0:45', '1:0', '1:15', '1:30', '1:45', '2:0']
    early_morning_timestamps = ['2:15', '2:30', '2:45', '3:0', '3:15', '3:30', '3:45', '4:0']
    
    df_48_morning = train[(train['day'] == 48) & (train['timestamp'].isin(morning_timestamps))].copy()
    df_49_morning = train[(train['day'] == 49) & (train['timestamp'].isin(morning_timestamps))].copy()
    df_48_early = train[(train['day'] == 48) & (train['timestamp'].isin(early_morning_timestamps))].copy()
    df_49_early = train[(train['day'] == 49) & (train['timestamp'].isin(early_morning_timestamps))].copy()
    
    df_48_morning['RoadType'] = df_48_morning['RoadType'].fillna('Missing')
    df_49_morning['RoadType'] = df_49_morning['RoadType'].fillna('Missing')
    df_48_early['RoadType'] = df_48_early['RoadType'].fillna('Missing')
    df_49_early['RoadType'] = df_49_early['RoadType'].fillna('Missing')
    
    mean_48 = df_48_morning.groupby('geohash')['demand'].mean()
    mean_49 = df_49_morning.groupby('geohash')['demand'].mean()
    
    trend_ratio = mean_49 / (mean_48 + 1e-5)
    trend_ratio = trend_ratio.rename('day49_trend_ratio')
    
    # Additional early morning trend for better test set alignment
    mean_48_early = df_48_early.groupby('geohash')['demand'].mean()
    mean_49_early = df_49_early.groupby('geohash')['demand'].mean()
    early_trend_ratio = mean_49_early / (mean_48_early + 1e-5)
    early_trend_ratio = early_trend_ratio.rename('early_morning_trend_ratio')
    
    # Group-level fallback ratios by RoadType
    roadtype_ratio_48 = df_48_morning.groupby('RoadType')['demand'].mean()
    roadtype_ratio_49 = df_49_morning.groupby('RoadType')['demand'].mean()
    roadtype_ratio = (roadtype_ratio_49 / (roadtype_ratio_48 + 1e-5)).rename('roadtype_trend_ratio')
    
    # Global fallback ratio
    global_ratio = df_49_morning['demand'].mean() / (df_48_morning['demand'].mean() + 1e-5)
    
    # Map trend ratio back to full_df
    full_df = pd.merge(full_df, trend_ratio, on='geohash', how='left')
    full_df = pd.merge(full_df, early_trend_ratio, on='geohash', how='left')
    full_df = pd.merge(full_df, roadtype_ratio.reset_index(), on='RoadType', how='left')
    
    # For Day 48 rows, set trend ratio to 1.0 (since there is no comparison)
    full_df.loc[full_df['day'] == 48, 'day49_trend_ratio'] = 1.0
    full_df.loc[full_df['day'] == 48, 'early_morning_trend_ratio'] = 1.0
    full_df['day49_trend_ratio'] = full_df['day49_trend_ratio'].fillna(full_df['roadtype_trend_ratio'])
    full_df['day49_trend_ratio'] = full_df['day49_trend_ratio'].fillna(global_ratio)
    full_df['early_morning_trend_ratio'] = full_df['early_morning_trend_ratio'].fillna(full_df['roadtype_trend_ratio'])
    full_df['early_morning_trend_ratio'] = full_df['early_morning_trend_ratio'].fillna(global_ratio)
    full_df['roadtype_trend_ratio'] = full_df['roadtype_trend_ratio'].fillna(global_ratio)
    full_df['geohash_day48_mean'] = full_df['geohash_day48_mean'].fillna(global_day48_mean)
    
    # Clip extreme ratios to prevent huge outlier swings
    full_df['day49_trend_ratio'] = full_df['day49_trend_ratio'].clip(0.1, 5.0)
    full_df['early_morning_trend_ratio'] = full_df['early_morning_trend_ratio'].clip(0.1, 5.0)
    
    # 5. Missing Value Imputation
    print("Imputing missing values...")
    full_df['RoadType'] = full_df['RoadType'].fillna('Missing')
    full_df['Weather'] = full_df['Weather'].fillna('Missing')
    
    # Median temperature per Weather
    weather_temp_medians = full_df.groupby('Weather')['Temperature'].transform('median')
    full_df['Temperature'] = full_df['Temperature'].fillna(weather_temp_medians)
    full_df['Temperature'] = full_df['Temperature'].fillna(full_df['Temperature'].median())
    
    # 6. Categorical Encoding
    print("Encoding categorical columns...")
    full_df['LargeVehicles'] = full_df['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(0).astype(int)
    full_df['Landmarks'] = full_df['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
    
    # Label encode categorical columns for LightGBM
    for col in ['RoadType', 'Weather', 'geohash']:
        full_df[col] = full_df[col].astype('category')
        
    # Split back into train and test
    train_preprocessed = full_df[full_df['is_test'] == 0].drop(columns=['is_test', 'Index']).reset_index(drop=True)
    test_preprocessed = full_df[full_df['is_test'] == 1].drop(columns=['is_test', 'demand']).reset_index(drop=True)
    
    # Define features to use
    features = [
        'latitude', 'longitude', 'minutes', 'hour',
        'sin_time', 'cos_time', 'is_peak_hour', 'is_night', 'is_morning', 'is_afternoon', 'is_evening',
        'demand_last_day', 'geohash_day48_mean', 'timestamp_day48_mean', 'timestamp_day48_std', 'roadtype_day48_mean',
        'day49_trend_ratio', 'early_morning_trend_ratio', 'roadtype_trend_ratio', 'day',
        'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
        'Temperature', 'Weather', 'geohash'
    ]
    
    target = 'demand'
    
    print(f"Preprocessing completed. Train shape: {train_preprocessed.shape}, Test shape: {test_preprocessed.shape}")
    return train_preprocessed, test_preprocessed, features, target

# Verification run
train_df, test_df, features, target = load_and_preprocess_data()
print('\nFeatures for training:', features)

## 3. 5-Fold Cross-Validation and Model Training
We set up a robust Out-of-Fold (OOF) cross-validation framework to prevent data leakage and train 5 LightGBM Regressors. CatBoost and XGBoost hyperparameters can also be optimized here.

In [ ]:
def train_model():
    train_df, test_df, features, target = load_and_preprocess_data()

    categorical_features = ['RoadType', 'Weather', 'geohash']

    print("\n--- Starting Training ---")
    print(f"Features used ({len(features)}): {features}")

    n_splits = 5
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    oof_preds_lgb = np.zeros(len(train_df))
    oof_preds_cat = np.zeros(len(train_df))
    oof_predictions = np.zeros(len(train_df))
    test_predictions = np.zeros(len(test_df))

    lgb_rmses = []
    cat_rmses = []
    ensemble_rmses = []
    lgb_maes = []
    cat_maes = []
    ensemble_maes = []

    os.makedirs("models", exist_ok=True)

    for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
        print(f"\n--- Fold {fold + 1} / {n_splits} ---")

        X_train = train_df.loc[train_idx, features]
        y_train = train_df.loc[train_idx, target]
        X_val = train_df.loc[val_idx, features]
        y_val = train_df.loc[val_idx, target]

        # LightGBM training
        lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_features)
        lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=categorical_features)

        lgb_params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'learning_rate': 0.02,
            'num_leaves': 50,
            'max_depth': 6,
            'min_child_samples': 30,
            'subsample': 0.7,
            'colsample_bytree': 0.7,
            'reg_alpha': 0.5,
            'reg_lambda': 1.0,
            'random_state': 42 + fold,
            'verbose': -1,
            'n_jobs': -1
        }

        lgb_model = lgb.train(
            lgb_params,
            lgb_train,
            valid_sets=[lgb_train, lgb_val],
            callbacks=[
                lgb.early_stopping(stopping_rounds=200, verbose=False),
                lgb.log_evaluation(period=200)
            ],
            num_boost_round=3500,
        )

        val_preds_lgb = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
        test_preds_lgb = lgb_model.predict(test_df[features], num_iteration=lgb_model.best_iteration)

        # CatBoost training
        cat_train = Pool(X_train, y_train, cat_features=categorical_features)
        cat_val = Pool(X_val, y_val, cat_features=categorical_features)

        cat_model = CatBoostRegressor(
            iterations=3500,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=3,
            border_count=128,
            bagging_temperature=0.4,
            random_seed=42 + fold,
            loss_function='RMSE',
            eval_metric='RMSE',
            early_stopping_rounds=200,
            verbose=200,
            thread_count=-1,
        )

        cat_model.fit(cat_train, eval_set=cat_val, use_best_model=True)

        val_preds_cat = cat_model.predict(X_val)
        test_preds_cat = cat_model.predict(test_df[features])

        # Ensemble average of CatBoost and LightGBM
        val_preds_ensemble = 0.5 * val_preds_lgb + 0.5 * val_preds_cat
        test_predictions += (test_preds_lgb + test_preds_cat) / (2 * n_splits)

        oof_preds_lgb[val_idx] = val_preds_lgb
        oof_preds_cat[val_idx] = val_preds_cat
        oof_predictions[val_idx] = val_preds_ensemble

        lgb_rmse = np.sqrt(mean_squared_error(y_val, val_preds_lgb))
        cat_rmse = np.sqrt(mean_squared_error(y_val, val_preds_cat))
        ensemble_rmse = np.sqrt(mean_squared_error(y_val, val_preds_ensemble))

        lgb_mae = mean_absolute_error(y_val, val_preds_lgb)
        cat_mae = mean_absolute_error(y_val, val_preds_cat)
        ensemble_mae = mean_absolute_error(y_val, val_preds_ensemble)

        lgb_rmses.append(lgb_rmse)
        cat_rmses.append(cat_rmse)
        ensemble_rmses.append(ensemble_rmse)
        lgb_maes.append(lgb_mae)
        cat_maes.append(cat_mae)
        ensemble_maes.append(ensemble_mae)

        print(f"Fold {fold + 1} LGB RMSE: {lgb_rmse:.6f} | CAT RMSE: {cat_rmse:.6f} | ENS RMSE: {ensemble_rmse:.6f}")
        print(f"Fold {fold + 1} LGB MAE: {lgb_mae:.6f} | CAT MAE: {cat_mae:.6f} | ENS MAE: {ensemble_mae:.6f}")

        lgb_model_path = f"models/lgb_fold_{fold + 1}.pkl"
        cat_model_path = f"models/catboost_fold_{fold + 1}.pkl"

        with open(lgb_model_path, 'wb') as f:
            pickle.dump(lgb_model, f)

        with open(cat_model_path, 'wb') as f:
            pickle.dump(cat_model, f)

    overall_rmse_lgb = np.sqrt(mean_squared_error(train_df[target], oof_preds_lgb))
    overall_rmse_cat = np.sqrt(mean_squared_error(train_df[target], oof_preds_cat))
    overall_rmse_ens = np.sqrt(mean_squared_error(train_df[target], oof_predictions))
    overall_mae_lgb = mean_absolute_error(train_df[target], oof_preds_lgb)
    overall_mae_cat = mean_absolute_error(train_df[target], oof_preds_cat)
    overall_mae_ens = mean_absolute_error(train_df[target], oof_predictions)
    overall_r2 = r2_score(train_df[target], oof_predictions)

    print("\n--- Out-Of-Fold Summary ---")
    print(f"LightGBM RMSE: {np.mean(lgb_rmses):.6f} Â± {np.std(lgb_rmses):.6f}")
    print(f"CatBoost RMSE: {np.mean(cat_rmses):.6f} Â± {np.std(cat_rmses):.6f}")
    print(f"Ensemble RMSE: {np.mean(ensemble_rmses):.6f} Â± {np.std(ensemble_rmses):.6f}")
    print(f"Overall OOF RMSE: {overall_rmse_ens:.6f}")
    print(f"Overall OOF MAE: {overall_mae_ens:.6f}")
    print(f"Overall OOF R2 Score: {overall_r2:.6f}")

    print("\nSaving ensemble predictions...")
    test_df_out = test_df.copy()
    test_df_out['demand'] = test_predictions.clip(0.0, 1.0)

    os.makedirs("predictions", exist_ok=True)
    test_df_out.to_csv("predictions/test_predictions.csv", index=False)

    submission = test_df_out[['Index', 'demand']]
    submission.to_csv("submission.csv", index=False)
    print("Submission file generated successfully!")

    return oof_predictions, test_predictions

# Run the complete training loop!
oof_preds, test_preds = train_model()

## 4. Model Interpretation & Feature Importance
Let's look at which features are most predictive for traffic demand forecasting:

In [ ]:
# Load one of the trained fold models to plot feature importance
with open('models/lgb_fold_1.pkl', 'rb') as f:
    model = pickle.load(f)

importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance['Feature'], importance['Importance'], color='teal')
plt.xlabel('Gain Importance')
plt.title('LightGBM Feature Importance')
plt.tight_layout()
plt.show()

## 5. Final Submission Verification
Let's verify that the output submission file meets all format requirements (correct row count, non-negative demand values, non-null values):

In [ ]:
sub = pd.read_csv('submission.csv')
print('--- Submission Head ---')
print(sub.head())

print('\n--- Submission Info ---')
print(sub.info())

print('\nNull count:', sub.isnull().sum().sum())
print('Min demand:', sub['demand'].min())
print('Max demand:', sub['demand'].max())
print('Number of rows matching test set:', len(sub) == 41778)